# DEPRECATED — NON-CANONICAL
Do not execute this historical evaluator. Start only with `00_campaign_control.ipynb` and follow `docs/canonical-colab-runbook.md`.

# EdgeGuard-Road single-scale PIDNet-S Cityscapes-val evaluation
Execution-only notebook for the reviewed full-resolution project evaluation. It does not claim reproduction of the official PIDNet paper protocol, OOD performance, threshold quality, or anomaly probability.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
RUN_ENV = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}
REPOSITORY_URL = "https://github.com/emrealmaoglu/edgeguard-road.git"
BRANCH = "feat/first-vertical-slice"
EXPECTED_COMMIT = input("Reviewed Commit C SHA: ").strip()
REPO_ROOT = Path("/content/edgeguard-road")
if len(EXPECTED_COMMIT) != 40 or any(c not in "0123456789abcdef" for c in EXPECTED_COMMIT):
    raise RuntimeError("Enter the reviewed lowercase 40-character Commit C SHA")
if REPO_ROOT.exists():
    raise RuntimeError("Start from a fresh Colab runtime")
subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY_URL, str(REPO_ROOT)],
    check=True,
    env=RUN_ENV,
)
actual_commit = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
).stdout.strip()
if actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(f"Commit mismatch: expected {EXPECTED_COMMIT}, got {actual_commit}")
os.chdir(REPO_ROOT)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[dev]"], cwd=REPO_ROOT, check=True, env=RUN_ENV
)
subprocess.run(
    [sys.executable, "-m", "edgeguard", "doctor", "--json"], cwd=REPO_ROOT, check=True, env=RUN_ENV
)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_ROOT, check=True, env=RUN_ENV)
status = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
    env=RUN_ENV,
).stdout.strip()
if status:
    raise RuntimeError(f"Repository became dirty: {status}")

## Human-controlled private inputs
Mount private Drive storage, then enter runtime paths. Checkpoint and Cityscapes archives remain outside Git and are verified by their pinned hashes before use.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CHECKPOINT = Path(input("Private Drive checkpoint path: ").strip())
LEFT_ARCHIVE = Path(input("Private Drive leftImg8bit archive path: ").strip())
LABEL_ARCHIVE = Path(input("Private Drive gtFine archive path: ").strip())
CHECKPOINT_ACCESS_DATE = input("Checkpoint access date YYYY-MM-DD: ").strip()
SAMPLE_ACCESS_DATE = input("Upstream sample access date YYYY-MM-DD: ").strip()
PIDNET_COMMIT = "4c158cf24ce432f0a8cb43364fae38d93cee0dc3"
PIDNET_CHECKOUT = REPO_ROOT / "artifacts/external/pidnet" / PIDNET_COMMIT
PIDNET_CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
CITYSCAPES_ROOT = Path("/content/cityscapes-val")
OUTPUT_ROOT = Path("/content/edgeguard-cityscapes-eval")
subprocess.run(
    [
        "git",
        "clone",
        "--no-checkout",
        "https://github.com/XuJiacong/PIDNet.git",
        str(PIDNET_CHECKOUT),
    ],
    check=True,
    env=RUN_ENV,
)
subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "checkout", "--detach", PIDNET_COMMIT],
    check=True,
    env=RUN_ENV,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/verify_pidnet_checkpoint.py",
        "--config",
        "configs/pidnet_spike.yaml",
        "--upstream-checkout",
        str(PIDNET_CHECKOUT),
        "--checkpoint",
        str(CHECKPOINT),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
subprocess.run(
    [
        sys.executable,
        "scripts/prepare_cityscapes.py",
        "--left-images-archive",
        str(LEFT_ARCHIVE),
        "--labels-archive",
        str(LABEL_ARCHIVE),
        "--destination",
        str(CITYSCAPES_ROOT),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/run_pidnet_spike.py",
        "--config",
        "configs/pidnet_spike.yaml",
        "--upstream-checkout",
        str(PIDNET_CHECKOUT),
        "--checkpoint",
        str(CHECKPOINT),
        "--checkpoint-access-date",
        CHECKPOINT_ACCESS_DATE,
        "--sample-access-date",
        SAMPLE_ACCESS_DATE,
        "--device",
        "cuda",
        "--output-dir",
        "artifacts/dev/pidnet_spike_clean_colab",
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
subprocess.run(
    [
        sys.executable,
        "scripts/run_cityscapes_eval.py",
        "--config",
        "configs/cityscapes_eval_colab.yaml",
        "--dataset-root",
        str(CITYSCAPES_ROOT),
        "--checkpoint",
        str(CHECKPOINT),
        "--upstream-checkout",
        str(PIDNET_CHECKOUT),
        "--split",
        "val",
        "--subset-size",
        "1",
        "--device",
        "cuda",
        "--output-dir",
        str(OUTPUT_ROOT / "preflight-1"),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)

In [ ]:
from google.colab import files

FULL_OUTPUT = OUTPUT_ROOT / "full-500"
subprocess.run(
    [
        sys.executable,
        "scripts/run_cityscapes_eval.py",
        "--config",
        "configs/cityscapes_eval_colab.yaml",
        "--dataset-root",
        str(CITYSCAPES_ROOT),
        "--checkpoint",
        str(CHECKPOINT),
        "--upstream-checkout",
        str(PIDNET_CHECKOUT),
        "--split",
        "val",
        "--all",
        "--device",
        "cuda",
        "--output-dir",
        str(FULL_OUTPUT),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
PACKAGE = Path("/content/edgeguard-cityscapes-eval.zip")
subprocess.run(
    [
        sys.executable,
        "scripts/package_eval_artifacts.py",
        "--input-dir",
        str(FULL_OUTPUT),
        "--output-zip",
        str(PACKAGE),
    ],
    cwd=REPO_ROOT,
    check=True,
    env=RUN_ENV,
)
files.download(str(PACKAGE))